# Chapter 17: QAE

---

**Prerequisites:**
- See `Chapter02_QuantumSoftware.ipynb` for installation instructions


In [1]:
# Setup and imports
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler 
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from Chapter03_EngineeringOptimization_functions import (truss2x2,truss2x3,truss3x3)

from Chapter15_QAE_functions import build_grover_operator, build_observable_circuit

In [2]:
beta   = np.pi / 6
a_true = np.sin(beta)**2   # 0.25

epsilon_target = 0.001
# State preparation: A = R_y(2 beta)
A = QuantumCircuit(1)
A.ry(2*beta, 0)

# Good state = |1> on qubit 0
problem = EstimationProblem(state_preparation=A, objective_qubits=[0])

# Target precision 0.01 on a, 95% confidence
iqae = IterativeAmplitudeEstimation(epsilon_target=epsilon_target,alpha=0.05,sampler=StatevectorSampler())
result = iqae.estimate(problem)

print(f"True a:            {a_true:.4f}")
print(f"IQAE estimate:     {result.estimation:.4f}")
print(f"95% CI:            [{result.confidence_interval[0]:.4f}, "
      f"{result.confidence_interval[1]:.4f}]")
print(f"Oracle queries:    {result.num_oracle_queries}")
print(f"k schedule:        {result.powers}")

True a:            0.2500
IQAE estimate:     0.2500
95% CI:            [0.2498, 0.2502]
Oracle queries:    104448
k schedule:        [0, 0, 6, 96]


### Observable

In [3]:
# --- Problem definition ---
f = np.array([1., 0., 0.5, 0.2])
f = f / np.linalg.norm(f)
A_mat = np.array([[ 1.,  0.,  0., -0.5],
                  [ 0.,  1.,  0.,  0. ],
                  [ 0.,  0.,  1.,  0. ],
                  [-0.5, 0.,  0.,  1. ]])
x_vec     = np.array([0.6, 0.8, 0., 0.])
classical = np.abs(f @ A_mat @ x_vec)

# --- Build circuit ---
A_obs, metadata = build_observable_circuit(A_mat, x_vec, f)
alpha_lcu  = metadata['alpha']
p_succ     = metadata['p_success']
good_qubits = metadata['good_qubits']

# --- Statevector diagnostics ---
num_anc = metadata['num_ancilla']
num_sys = metadata['num_system']
sv = Statevector(A_obs)
probs = sv.probabilities_dict()


# --- Run IQAE ---
problem = EstimationProblem(state_preparation=A_obs, objective_qubits=good_qubits)
iqae    = IterativeAmplitudeEstimation(epsilon_target=0.01, alpha=0.05,
                                       sampler=StatevectorSampler())
result  = iqae.estimate(problem)

a_hat, (a_lo, a_hi) = result.estimation, result.confidence_interval

# --- Recovery formula ---
obs_hat = np.sqrt(a_hat) * alpha_lcu * np.sqrt(p_succ)
obs_lo  = np.sqrt(a_lo)  * alpha_lcu * np.sqrt(p_succ)
obs_hi  = np.sqrt(a_hi)  * alpha_lcu * np.sqrt(p_succ)

print("=== Results ===")
print(f"Classical:           {classical:.4f}")
print(f"IQAE estimate:       {obs_hat:.4f}")
print(f"95% CI:              [{obs_lo:.4f}, {obs_hi:.4f}]")
print(f"Oracle queries:      {result.num_oracle_queries}")
print(f"k schedule:        {result.powers}")


=== Results ===
Classical:           0.4754
IQAE estimate:       0.4755
95% CI:              [0.4729, 0.4781]
Oracle queries:      7168
k schedule:        [0, 0, 7]
